# 10 — Tokenization — Solution

---

In [ ]:
import sys, os, re
sys.path.insert(0, os.path.abspath('../..'))
from collections import Counter, defaultdict
import torch, matplotlib.pyplot as plt

print('Imports OK')

## Part B — BPE from Scratch

In [ ]:
def get_vocab(corpus):
    vocab = defaultdict(int)
    for word in corpus:
        chars = tuple(list(word) + ['</w>'])
        vocab[chars] += 1
    return dict(vocab)


def get_pairs(vocab):
    pairs = Counter()
    for word, freq in vocab.items():
        for i in range(len(word) - 1):
            pairs[(word[i], word[i+1])] += freq   # weight by word frequency
    return pairs


def merge_vocab(pair, vocab):
    new_vocab = {}
    # Build a regex that matches the pair at word boundaries
    bigram = re.escape(' '.join(pair))
    pattern = re.compile(r'(?<!\S)' + bigram + r'(?!\S)')
    for word, freq in vocab.items():
        word_str = ' '.join(word)
        new_word_str = pattern.sub(''.join(pair), word_str)
        new_vocab[tuple(new_word_str.split())] = freq
    return new_vocab


corpus = "low low low low lowest lowest newer newer newer newer wider wider new new".split()
vocab = get_vocab(corpus)
print('Initial vocab:')
for w, f in sorted(vocab.items(), key=lambda x: -x[1]):
    print(f'  {" ".join(w):<20} : {f}')

print('\nBPE merges:')
for i in range(10):
    pairs = get_pairs(vocab)
    if not pairs: break
    best = pairs.most_common(1)[0][0]
    vocab = merge_vocab(best, vocab)
    print(f'  Merge {i+1:2d}: {best[0]} + {best[1]} → {"" .join(best)}')

## Parts C–E — Tokenizer & Padding

In [ ]:
try:
    from src.data.tokenizer import BPETokenizer
    tokenizer = BPETokenizer()
    texts = [
        "once upon a time in a land far far away",
        "the transformer architecture revolutionised natural language processing",
        "attention is all you need to build powerful language models",
    ] * 10
    tokenizer.train(texts, vocab_size=200, model_prefix='workshop_tok')
    test = "the transformer learns to attend to relevant tokens"
    ids = tokenizer.encode(test)
    print(f'Input    : {test}')
    print(f'Token IDs: {ids}')
    print(f'Decoded  : {tokenizer.decode(ids)}')
    print(f'Vocab size: {tokenizer.vocab_size}')
except ImportError:
    print('sentencepiece not available — skipping.')

In [ ]:
# Vocab size vs sequence length
sample_text = "the quick brown fox jumps over the lazy dog " * 5
words = sample_text.split()
char_len = len(sample_text.replace(' ', ''))
word_len  = len(words)
subword   = int(word_len * 1.3)

methods  = ['Character\n(vocab~100)', 'BPE Subword\n(vocab~50k)', 'Word\n(vocab~200k)']
seq_lens = [char_len, subword, word_len]

plt.figure(figsize=(7, 4))
plt.bar(methods, seq_lens, color=['tomato', 'steelblue', 'seagreen'])
plt.ylabel('Sequence length'); plt.title('Tokenisation: Sequence Length vs Method')
for i, v in enumerate(seq_lens): plt.text(i, v+1, str(v), ha='center')
plt.tight_layout(); plt.show()

In [ ]:
# Padding and attention masks
sequences = [[10,24,7,3], [5,18,2,41,9,15], [33,7], [12,4,8,27,3]]
PAD = 0
max_len = max(len(s) for s in sequences)

padded    = [s + [PAD] * (max_len - len(s)) for s in sequences]
attn_mask = [[1]*len(s) + [0]*(max_len-len(s)) for s in sequences]

padded    = torch.tensor(padded)
attn_mask = torch.tensor(attn_mask)

print('Padded:'); print(padded)
print('\nMask (1=real):'); print(attn_mask)